# Output conditioning

This notebook is part of the staged manuscript-reproduction workflow. During automated validation it resolves placeholder paths to the deterministic demo dataset; once local real-data paths are configured it will run against the real case-study artifacts.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    env_root = os.environ.get("BSM_PUBLIC_RF_REPO_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not locate the bsm-public-rf repository root.")


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from bsm_rfm import (
    build_manuscript_notebook_context,
    run_output_conditioning_stage,
)

context = build_manuscript_notebook_context(REPO_ROOT, "02_output_conditioning.ipynb")
stage_result = run_output_conditioning_stage(context)
summary = stage_result.conditioning.summary
summary

### Step 1: Apply Conditioning Rules

Apply data transformations and filters from configuration.

**Operations:**
- Drop rows with any missing values (or use configured strategy)
- Apply scaling if specified (z-score, robust, or none)
- Apply final exclusion filters (zero-variance after transformation, etc.)
- Validate output schema (no NaN, proper dtypes)

**Output:**
- `dfm_conditioned`: Cleaned, scaled feature DataFrame ready for modeling

### Step 2: Generate Data Quality Report

Document conditioning results and save handoff CSV.

**Report contents:**
- Before/after dimensions (samples, features)
- Missing data statistics
- Scaling parameters (if applied)
- Final feature list with metadata